> **Provenance des données** — Les fichiers produits par `utils_louisfarm.py` sont synthétiques et reproductibles. Les noms de pays contextualisent les exercices ; les observations ne proviennent pas d’une enquête ni d’une institution financière réelle. Les résultats ne décrivent pas les populations de ces pays.
> Pour votre projet, documentez la source, la date, les unités et les droits d’utilisation. Les données personnelles doivent être anonymisées.


# LouisFarm — Semaine 6 : Prediction & Regression
## Dataset : Immobilier Abidjan (2000 transactions)

**Objectif :** Construire des modeles predictifs de regression rigoureux.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns; sns.set_theme(style="whitegrid")
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings; warnings.filterwarnings("ignore")
import sys; sys.path.insert(0, ".")
from utils_louisfarm import gen_immobilier_abidjan

df = gen_immobilier_abidjan(n=2000)
print(f"Dataset Immobilier Abidjan: {df.shape}")
print(df.head(4))
print(f"\nPrix median: {df.prix_fcfa.median():,.0f} FCFA")
print(f"Surface med: {df.surface_m2.median():.0f} m2")

## Lecon 6.1 — EDA avant modelisation

In [ ]:
# EDA avant modelisation
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Semaine 6 - EDA Immobilier Abidjan avant modelisation", fontweight="bold")

# Distribution prix
axes[0,0].hist(df.prix_fcfa/1e6, bins=40, color="#2E86AB", alpha=0.8)
axes[0,0].set_xlabel("Prix (millions FCFA)"); axes[0,0].set_title("Distribution des prix")

# Log prix (transformation)
axes[0,1].hist(np.log(df.prix_fcfa), bins=40, color="#A23B72", alpha=0.8)
axes[0,1].set_xlabel("Log(Prix FCFA)"); axes[0,1].set_title("Distribution log-prix (plus normale)")

# Surface vs prix
axes[0,2].scatter(df.surface_m2, df.prix_fcfa/1e6, alpha=0.3, s=10, color="#F18F01")
axes[0,2].set_xlabel("Surface (m2)"); axes[0,2].set_ylabel("Prix (M FCFA)")
axes[0,2].set_title("Surface vs Prix")

# Prix par commune
by_comm = df.groupby("commune")["prix_fcfa"].median().sort_values()/1e6
axes[1,0].barh(by_comm.index, by_comm.values, color="#2E86AB")
axes[1,0].set_xlabel("Prix median (M FCFA)"); axes[1,0].set_title("Prix median par commune")

# Prix par type
by_type = df.groupby("type_logement")["prix_fcfa"].median()/1e6
axes[1,1].bar(by_type.index, by_type.values, color=["#2E86AB","#A23B72","#F18F01","#C73E1D"])
axes[1,1].set_title("Prix median par type")
plt.setp(axes[1,1].get_xticklabels(), rotation=30, ha="right")

# Correlation heatmap
num_cols = ["surface_m2","nbr_chambres","etage","distance_centre_km","annee_construction","prix_fcfa"]
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1,2])
axes[1,2].set_title("Correlations")

plt.tight_layout()
plt.savefig("./s6_eda.png", dpi=100, bbox_inches="tight")
plt.show()

## Lecon 6.2 — Pipeline sklearn : LinearRegression & Ridge

In [ ]:
# PREPARATION DES DONNEES
df["log_prix"] = np.log(df["prix_fcfa"])
X = df[["surface_m2","nbr_chambres","etage","distance_centre_km",
        "annee_construction","parking","gardiennage","commune","type_logement","etat"]]
y = df["log_prix"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# PIPELINE SKLEARN
num_features = ["surface_m2","nbr_chambres","etage","distance_centre_km","annee_construction","parking","gardiennage"]
cat_features = ["commune","type_logement","etat"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
])

# Modele 1: Regression lineaire
pipe_lr = Pipeline([("prep", preprocessor), ("model", LinearRegression())])
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr  = r2_score(y_test, y_pred_lr)

# Modele 2: Ridge (regularisation L2)
pipe_ridge = Pipeline([("prep", preprocessor), ("model", Ridge(alpha=10.0))])
pipe_ridge.fit(X_train, y_train)
y_pred_ridge = pipe_ridge.predict(X_test)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
r2_ridge  = r2_score(y_test, y_pred_ridge)

# Baseline: predire la moyenne
y_baseline = np.full(len(y_test), y_train.mean())
mae_base = mean_absolute_error(y_test, y_baseline)
r2_base  = r2_score(y_test, y_baseline)

print("\nCOMPARAISON DES MODELES (log-prix)")
print("=" * 52)
print(f"  Baseline (moyenne) : MAE={mae_base:.4f} | R2={r2_base:.4f}")
print(f"  LinearRegression   : MAE={mae_lr:.4f} | R2={r2_lr:.4f}")
print(f"  Ridge (alpha=10)   : MAE={mae_ridge:.4f} | R2={r2_ridge:.4f}")

# Calculer la MAE dans l’unité d’origine, après transformation inverse
mae_fcfa_lr    = mean_absolute_error(np.exp(y_test), np.exp(y_pred_lr)) / 1e6
mae_fcfa_ridge = mean_absolute_error(np.exp(y_test), np.exp(y_pred_ridge)) / 1e6
print(f"\n  Erreur absolue moyenne (prix FCFA):")
print(f"    LinearRegression : ~{mae_fcfa_lr:.1f} millions FCFA")
print(f"    Ridge            : ~{mae_fcfa_ridge:.1f} millions FCFA")

## Lecon 6.3 — Visualisation et evaluation

In [ ]:
# VISUALISATION DES RESULTATS
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Semaine 6 - Evaluation des modeles de regression", fontweight="bold")

# Predit vs Reel
axes[0].scatter(y_test, y_pred_ridge, alpha=0.4, s=15, color="#2E86AB")
min_v, max_v = y_test.min(), y_test.max()
axes[0].plot([min_v,max_v],[min_v,max_v],"r--", linewidth=2, label="Prediction parfaite")
axes[0].set_xlabel("Log-prix reel"); axes[0].set_ylabel("Log-prix predit")
axes[0].set_title(f"Ridge : Predit vs Reel (R2={r2_ridge:.3f})")
axes[0].legend()

# Distribution des residus
residus = y_test - y_pred_ridge
axes[1].hist(residus, bins=40, color="#A23B72", alpha=0.8)
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Residus"); axes[1].set_title("Distribution des residus (Ridge)")

# Comparaison MAE
models = ["Baseline", "LinearReg", "Ridge"]
maes = [mae_base, mae_lr, mae_ridge]
colors_bar = ["#E8E8E8","#F18F01","#2E86AB"]
bars = axes[2].bar(models, maes, color=colors_bar, edgecolor="white")
axes[2].set_ylabel("MAE (log-prix)")
axes[2].set_title("Comparaison des modeles")
for bar, mae in zip(bars, maes):
    axes[2].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.0005,
                 f"{mae:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("./s6_models.png", dpi=100, bbox_inches="tight")
plt.show()
print("Sauvegarde: s6_models.png")

## Lecon 6.4 — Interpretation et deploiement

In [ ]:
# INTERPRETATION METIER
import joblib

# Sauvegarder le modele
joblib.dump(pipe_ridge, "./modele_prix_abidjan.pkl")
print("Modele sauvegarde: modele_prix_abidjan.pkl")

# Prediction sur un nouveau bien
nouveau_bien = pd.DataFrame({
    "surface_m2": [85],
    "nbr_chambres": [3],
    "etage": [2],
    "distance_centre_km": [5.0],
    "annee_construction": [2018],
    "parking": [1],
    "gardiennage": [1],
    "commune": ["Cocody"],
    "type_logement": ["Appartement"],
    "etat": ["Bon etat"],
})

log_prix_pred = pipe_ridge.predict(nouveau_bien)[0]
prix_pred = np.exp(log_prix_pred)
print(f"\nPREDICTION POUR UN APPARTEMENT 85 m2 A COCODY:")
print(f"  Prix estime : {prix_pred:>15,.0f} FCFA")
print(f"  Prix estime : {prix_pred/1e6:>10.1f} millions FCFA")
print(f"  Prix/m2     : {prix_pred/85:>12,.0f} FCFA/m2")

print("\n=== NOTE POUR LE DIRECTEUR COMMERCIAL ===")
print("Le modele Ridge predit les prix avec une precision de")
print(f"~{mae_fcfa_ridge:.1f} millions FCFA d erreur moyenne.")
print("Il surpasse la methode de prix moyen global de {:.0f}%.".format(
    (1 - mae_ridge/mae_base)*100))
print("Principaux facteurs : surface, commune, type de logement.")

## Exercices S6

1. Essayez alpha=0.1, 1, 10, 100 pour Ridge. Quel alpha minimise la MAE en validation croisée sur X_train ? Gardez X_test pour le bilan final.
2. Ajoutez une feature `age_bien = 2024 - annee_construction`. Ameliore-t-elle le modele ?
3. Le R2 sur train est-il tres different du R2 sur test ? Que cela indique-t-il ?

In [ ]:
# EXERCICE 6.1 — Sélection sur l’entraînement uniquement
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.base import clone
search = GridSearchCV(
    Pipeline([("prep", clone(preprocessor)), ("model", Ridge())]),
    {"model__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
    scoring="neg_mean_absolute_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
)
search.fit(X_train, y_train)
print("Alpha retenu :", search.best_params_["model__alpha"])
print("MAE validation (log-prix) :", -search.best_score_)
# Une seule évaluation finale ; ne plus ajuster alpha après ce résultat.
final_predictions = search.predict(X_test)
print("MAE test (FCFA) :", mean_absolute_error(np.exp(y_test), np.exp(final_predictions)))
